# Input Maps, Target Climatology & Obs vs Pred

1. **Auxiliary static inputs** — all `vars_aux` channels
2. **Level input snapshot** — ta, hus, ua, va at 700 hPa (first sample)
3. **Target climatology** — time-mean total cloud cover
4. **Observed vs Predicted** — mean maps, bias, and scatter

In [ ]:
VAL_ZARR_PATH = "../data/train_data/val_era5_region1.zarr"
EXP_NAME = "baseline_mlp"
MODEL_LABEL = "MLP Baseline"
FORECAST_PATH = f"../data/forecasts/{EXP_NAME}/forecast.nc"

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

In [ ]:
ds_val = xr.open_zarr(VAL_ZARR_PATH)

# Normalize longitudes from [0, 360) to [-180, 180) for Cartopy
lon_norm = ((ds_val.lon.values + 180) % 360) - 180
ds_val = ds_val.assign_coords(lon=lon_norm).sortby("lon")

lat = ds_val.lat.values
lon = ds_val.lon.values
ds_val

In [ ]:
PROJ = ccrs.PlateCarree()


def map_ax(ax, data, lat, lon, cmap, vmin=None, vmax=None, cbar_label=""):
    """Filled pcolormesh map with coastlines on a Cartopy axes."""
    im = ax.pcolormesh(
        lon, lat, data, transform=PROJ, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto"
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":")
    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=PROJ)
    plt.colorbar(
        im, ax=ax, orientation="horizontal", pad=0.04, shrink=0.9, label=cbar_label
    )
    return im

## 1. Auxiliary static inputs

In [ ]:
aux = ds_val.input_auxiliary.compute()
vars_aux = aux.vars_aux.values

ncols = 4
nrows = int(np.ceil(len(vars_aux) / ncols))

fig, axes = plt.subplots(
    nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), subplot_kw={"projection": PROJ}
)
axes = np.array(axes).flatten()

for i, var in enumerate(vars_aux):
    data = aux.sel(vars_aux=var).values
    map_ax(axes[i], data, lat, lon, cmap="viridis", cbar_label=var)
    axes[i].set_title(var, fontsize=9)

for ax in axes[len(vars_aux) :]:
    ax.set_visible(False)

fig.suptitle("Auxiliary (static) inputs", fontsize=12)
plt.tight_layout()
plt.show()

## 2. Level input snapshot (700 hPa, first sample)

In [ ]:
LEVEL = 700
SAMPLE_IDX = 0

LONG_NAMES = {
    "ta": "Air temperature (K)",
    "hus": "Specific humidity (kg kg⁻¹)",
    "ua": "Zonal wind (m s⁻¹)",
    "va": "Meridional wind (m s⁻¹)",
}
CMAPS = {"ta": "RdBu_r", "hus": cmocean.cm.haline, "ua": "RdBu_r", "va": "RdBu_r"}

snap = ds_val.input_level.isel(time=SAMPLE_IDX).sel(level=LEVEL).compute()

fig, axes = plt.subplots(2, 2, figsize=(12, 7), subplot_kw={"projection": PROJ})
axes = axes.flatten()

for i, var in enumerate(LONG_NAMES):
    data = snap.sel(vars_level=var).values
    vabs = float(np.nanpercentile(np.abs(data), 99))
    if var in ("ua", "va"):
        map_ax(
            axes[i],
            data,
            lat,
            lon,
            cmap=CMAPS[var],
            vmin=-vabs,
            vmax=vabs,
            cbar_label=LONG_NAMES[var],
        )
    else:
        map_ax(axes[i], data, lat, lon, cmap=CMAPS[var], cbar_label=LONG_NAMES[var])
    axes[i].set_title(f"{LONG_NAMES[var]} — {LEVEL} hPa", fontsize=9)

fig.suptitle(f"Level inputs — sample {SAMPLE_IDX}, {LEVEL} hPa", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Target climatology (time-mean cloud cover)

In [ ]:
target_clim = ds_val.target.mean("time").compute()

fig, ax = plt.subplots(1, 1, figsize=(9, 5), subplot_kw={"projection": PROJ})
map_ax(
    ax,
    target_clim.values,
    lat,
    lon,
    cmap=cmocean.cm.gray_r,
    vmin=0,
    vmax=1,
    cbar_label="Cloud cover fraction",
)
ax.set_title("Validation target — mean total cloud cover (ERA5)", fontsize=11)
plt.tight_layout()
plt.show()

## 4. Observed vs Predicted

In [ ]:
ds_pred = xr.open_dataset(FORECAST_PATH)

# Normalize forecast lons to match
lon_pred_norm = ((ds_pred.lon.values + 180) % 360) - 180
ds_pred = ds_pred.assign_coords(lon=lon_pred_norm).sortby("lon")

obs = ds_val.target.compute()  # (time, lat, lon)
pred = ds_pred.total_cloud_cover.compute()  # (sample, lat, lon)

assert obs.shape == pred.shape, (
    f"Shape mismatch: obs {obs.shape} vs pred {pred.shape}. "
    "Ensure the forecast was generated from the same validation zarr."
)
print(f"obs: {obs.shape}   pred: {pred.shape}")

In [ ]:
obs_mean = obs.mean("time").values
pred_mean = pred.mean("sample").values
bias = pred_mean - obs_mean
vmax_bias = float(np.abs(bias).max())

fig, axes = plt.subplots(1, 3, figsize=(18, 5), subplot_kw={"projection": PROJ})

map_ax(
    axes[0],
    obs_mean,
    lat,
    lon,
    cmap=cmocean.cm.gray_r,
    vmin=0,
    vmax=1,
    cbar_label="Cloud cover fraction",
)
axes[0].set_title("Observed mean (ERA5)", fontsize=10)

map_ax(
    axes[1],
    pred_mean,
    lat,
    lon,
    cmap=cmocean.cm.gray_r,
    vmin=0,
    vmax=1,
    cbar_label="Cloud cover fraction",
)
axes[1].set_title(f"Predicted mean ({MODEL_LABEL})", fontsize=10)

map_ax(
    axes[2],
    bias,
    lat,
    lon,
    cmap="RdBu_r",
    vmin=-vmax_bias,
    vmax=vmax_bias,
    cbar_label="Pred − Obs",
)
axes[2].set_title(f"Bias ({MODEL_LABEL} − ERA5)", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
obs_flat = obs.values.ravel()
pred_flat = pred.values.ravel()

mae = float(np.abs(obs_flat - pred_flat).mean())

# Subsample for hexbin to keep rendering fast
rng = np.random.default_rng(42)
idx = rng.choice(len(obs_flat), size=min(100_000, len(obs_flat)), replace=False)

fig, ax = plt.subplots(figsize=(6, 6))
hb = ax.hexbin(
    obs_flat[idx],
    pred_flat[idx],
    gridsize=80,
    cmap="Blues",
    extent=[0, 1, 0, 1],
    mincnt=1,
)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="1:1")
plt.colorbar(hb, ax=ax, label="Count")
ax.set_xlabel("Observed cloud cover")
ax.set_ylabel(f"Predicted cloud cover ({MODEL_LABEL})")
ax.set_title(f"Obs vs Pred  |  MAE = {mae:.4f}")
ax.legend()
plt.tight_layout()
plt.show()